# TP01: Python Web Scraping Case-study
**Module:** Big Data Analytics — M2 IA & Data Science — جامعة الوادي 2026/2027

**الهدف:** جلب صفحات موقع تعليمي، استخراج الاقتباسات (Quotes)، حفظها في ملف CSV بواسطة pandas، والتأكد أن الملف يحتوي على **1000 سطر على الأقل**.

**خطة العمل:**
1. جلب كل صفحات `quotes.toscrape.com` (موقع مخصص للتدريب على الـ scraping) وعرض كود HTML.
2. استخراج الاقتباسات بـ BeautifulSoup.
3. هذا الموقع يحتوي على **100 اقتباس فقط** (10 صفحات × 10)، لذلك نضيف مصدراً ثانياً (Goodreads) للوصول إلى 1000+.
4. الحفظ في CSV والتحقق من الشرط.

## 0) المكتبات والإعدادات

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ترويسة User-Agent حتى لا يرفض الموقع الطلب
HEADERS = {"User-Agent": "Mozilla/5.0 (TP01 educational scraping - University of El Oued)"}

# عدد الأحرف المعروضة من كل صفحة HTML (ضع None لعرض الكود كاملاً)
PREVIEW = 1500

def get_html(url, retries=3, timeout=15):
    """تجلب محتوى صفحة مع إعادة المحاولة عند الفشل."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code == 200:
                r.encoding = "utf-8"
                return r.text
            print(f"[!] {url} -> HTTP {r.status_code} (محاولة {attempt})")
        except requests.RequestException as e:
            print(f"[!] خطأ في الاتصال بـ {url}: {e} (محاولة {attempt})")
        time.sleep(2)
    return None

## السؤال 1: جلب كل صفحات الموقع وعرض كود HTML
نبدأ من الصفحة الأولى ونتبع رابط **Next** (`li.next > a`) حتى لا يبقى رابط تالٍ.

In [ ]:
BASE = "https://quotes.toscrape.com"
pages_html = {}          # {رقم الصفحة: كود HTML}

url, page = BASE + "/", 1
while url:
    html = get_html(url)
    if html is None:
        print("توقف الجلب: تعذّر تحميل", url)
        break
    pages_html[page] = html

    print(f"\n===== الصفحة {page} | {url} | {len(html)} حرف =====")
    print(html if PREVIEW is None else html[:PREVIEW] + "\n...[تم الاقتطاع]...")

    soup = BeautifulSoup(html, "html.parser")
    nxt = soup.select_one("li.next > a")          # رابط الصفحة التالية
    url = BASE + nxt["href"] if nxt else None
    page += 1
    time.sleep(0.5)                                # تأخير بسيط احتراماً للخادم

print(f"\nعدد الصفحات المجلوبة: {len(pages_html)}")

## السؤال 2: استخراج الاقتباسات بـ BeautifulSoup
كل اقتباس داخل `div.quote` ويحتوي على: النص (`span.text`)، المؤلف (`small.author`)، والوسوم (`a.tag`).

In [ ]:
def parse_toscrape(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for q in soup.select("div.quote"):
        rows.append({
            "quote":  q.select_one("span.text").get_text(strip=True).strip("\u201c\u201d\" "),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags":   ", ".join(t.get_text(strip=True) for t in q.select("a.tag")),
            "source": "quotes.toscrape.com",
        })
    return rows

all_rows = []
for p, html in pages_html.items():
    rows = parse_toscrape(html)
    all_rows.extend(rows)
    print(f"الصفحة {p}: {len(rows)} اقتباس")

print(f"\nالمجموع من quotes.toscrape.com: {len(all_rows)} اقتباس")
pd.DataFrame(all_rows).head()

## إضافة مصدر ثانٍ للوصول إلى 1000+ سطر
الموقع الأول لا يتجاوز **100 اقتباس**، لذلك نجلب صفحات من `goodreads.com/quotes` (30 اقتباساً في الصفحة).
الدالة تتوقف تلقائياً عند الوصول للعدد المطلوب أو عند فشل الجلب.

In [ ]:
def parse_goodreads(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for q in soup.select("div.quoteDetails"):
        t = q.select_one("div.quoteText")
        if not t or not t.contents:
            continue
        text = str(t.contents[0]).strip().strip("\u201c\u201d\" ")      # أول نص قبل علامة ―
        a = q.select_one("span.authorOrTitle")
        author = a.get_text(strip=True).rstrip(",") if a else ""
        tags = ", ".join(x.get_text(strip=True) for x in q.select("div.greyText.smallText.left a"))
        if text:
            rows.append({"quote": text, "author": author, "tags": tags, "source": "goodreads.com"})
    return rows

TARGET = 1100          # هامش أمان فوق 1000 لأن بعض المكرر سيُحذف
MAX_PAGES = 60

for p in range(1, MAX_PAGES + 1):
    if len(all_rows) >= TARGET:
        break
    html = get_html(f"https://www.goodreads.com/quotes?page={p}")
    if html is None:
        print("توقف: تعذّر جلب الصفحة", p)
        break
    rows = parse_goodreads(html)
    if not rows:
        print(f"الصفحة {p}: لا اقتباسات (ربما تغيّر هيكل الموقع أو تم الحظر) -> توقف")
        break
    all_rows.extend(rows)
    print(f"Goodreads صفحة {p}: +{len(rows)} | الإجمالي = {len(all_rows)}")
    time.sleep(1.5)

## السؤال 3: الحفظ في ملف CSV بواسطة pandas

In [ ]:
df = pd.DataFrame(all_rows)
df = df.drop_duplicates(subset="quote").reset_index(drop=True)   # حذف المكرر
df.insert(0, "id", range(1, len(df) + 1))

# utf-8-sig حتى يظهر النص صحيحاً عند فتح الملف في Excel
df.to_csv("quotes.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ في quotes.csv")
df.head(10)

## السؤال 4: التحقق من 1000 سطر على الأقل

In [ ]:
check = pd.read_csv("quotes.csv", encoding="utf-8-sig")
print("عدد الأسطر:", len(check))
print(check["source"].value_counts(), "\n")

if len(check) >= 1000:
    print("✅ الشرط محقق: الملف يحتوي على 1000 سطر أو أكثر")
else:
    print(f"❌ ينقص {1000 - len(check)} سطر: ارفع MAX_PAGES أو TARGET وأعد تشغيل خلية Goodreads")

## (اختياري) تنزيل الملف ثم رفعه إلى GitHub
شغّل الخلية التالية لتنزيل `quotes.csv`، ثم ارفعه مع هذا الـ notebook إلى حسابك على GitHub (File ▸ Save a copy in GitHub).

In [ ]:
try:
    from google.colab import files
    files.download("quotes.csv")
except ImportError:
    print("لست في Colab: الملف محفوظ محلياً باسم quotes.csv")